In [1]:
import pandas as pd

df = pd.read_csv('filtered-data/muestra_ratings.csv')

In [4]:
import numpy as np
from sklearn.model_selection import train_test_split

from surprise import Dataset, Reader, KNNBasic

config = {
    'test_size': 0.2,
    'random_state': 42,
    'k_neighbors': 40,
    'min_k': 1,
    'user_caps': [8000, 6000, 4000, 2500],
    'max_test_eval': 20000,
}

train_df, test_df = train_test_split(
    df[['userId', 'movieId', 'rating']].copy(),
    test_size=config['test_size'],
    random_state=config['random_state']
 )

def _metrics_from_preds(y_true_arr, y_pred_arr, n_test):
    rmse_val = float(np.sqrt(np.mean((y_true_arr - y_pred_arr) ** 2))) if len(y_true_arr) else float('nan')
    mae_val = float(np.mean(np.abs(y_true_arr - y_pred_arr))) if len(y_true_arr) else float('nan')
    coverage_val = float(len(y_pred_arr) / n_test) if n_test else 0.0
    return rmse_val, mae_val, coverage_val

def _build_attempt_data(train_base_df, test_base_df, max_users, max_test_eval, random_state):
    user_counts = train_base_df['userId'].value_counts()
    if max_users is not None and len(user_counts) > max_users:
        keep_users = set(user_counts.head(max_users).index)
    else:
        keep_users = set(user_counts.index)

    train_attempt_df = train_base_df[train_base_df['userId'].isin(keep_users)].copy()

    train_users = set(train_attempt_df['userId'].unique())
    train_items = set(train_attempt_df['movieId'].unique())
    test_attempt_df = test_base_df[
        test_base_df['userId'].isin(train_users) &
        test_base_df['movieId'].isin(train_items)
    ].copy()

    if max_test_eval is not None and len(test_attempt_df) > max_test_eval:
        test_attempt_df = test_attempt_df.sample(n=max_test_eval, random_state=random_state)

    return train_attempt_df, test_attempt_df

trained_models = {}
model_results = []
attempt_logs = []

reader = Reader(rating_scale=(0.5, 5.0))

surprise_specs = [
    ('user_user_cosine', {'name': 'cosine', 'user_based': True}),
    ('user_user_pearson', {'name': 'pearson', 'user_based': True}),
    ('item_item_cosine', {'name': 'cosine', 'user_based': False}),
    ('item_item_pearson', {'name': 'pearson', 'user_based': False}),
]

for model_name, sim_options in surprise_specs:
    model_succeeded = False
    last_error = None

    for attempt_idx, user_cap in enumerate(config['user_caps'], start=1):
        train_attempt_df, test_attempt_df = _build_attempt_data(
            train_df,
            test_df,
            user_cap,
            config['max_test_eval'],
            config['random_state']
        )

        if len(train_attempt_df) == 0 or len(test_attempt_df) == 0:
            last_error = 'Empty train/test after filtering for this tier'
            attempt_logs.append({
                'model_name': model_name,
                'attempt': attempt_idx,
                'user_cap': user_cap,
                'status': 'failed',
                'train_ratings': int(len(train_attempt_df)),
                'test_ratings': int(len(test_attempt_df)),
                'error': last_error
            })
            continue

        y_true_attempt = np.array(test_attempt_df['rating'].astype(float).values, dtype=np.float32)

        try:
            train_data = Dataset.load_from_df(train_attempt_df[['userId', 'movieId', 'rating']], reader)
            trainset = train_data.build_full_trainset()

            algo_i = KNNBasic(
                k=config['k_neighbors'],
                min_k=config['min_k'],
                sim_options=sim_options,
                verbose=False
            )
            algo_i.fit(trainset)

            y_pred_i = np.array([
                float(algo_i.predict(uid=row.userId, iid=row.movieId, r_ui=row.rating).est)
                for row in test_attempt_df[['userId', 'movieId', 'rating']].itertuples(index=False)
            ], dtype=np.float32)

            rmse_i, mae_i, coverage_i = _metrics_from_preds(y_true_attempt, y_pred_i, len(test_attempt_df))

            trained_models[model_name] = algo_i
            attempt_logs.append({
                'model_name': model_name,
                'attempt': attempt_idx,
                'user_cap': user_cap,
                'status': 'success',
                'train_ratings': int(len(train_attempt_df)),
                'test_ratings': int(len(test_attempt_df)),
                'error': ''
            })

            model_results.append({
                'model_name': model_name,
                'status': 'success',
                'attempts_used': attempt_idx,
                'selected_user_cap': int(user_cap),
                'user_based': bool(sim_options['user_based']),
                'similarity': sim_options['name'],
                'train_users': int(train_attempt_df['userId'].nunique()),
                'train_items': int(train_attempt_df['movieId'].nunique()),
                'train_ratings': int(len(train_attempt_df)),
                'test_ratings': int(len(test_attempt_df)),
                'rmse': rmse_i,
                'mae': mae_i,
                'coverage': coverage_i,
                'last_error': ''
            })
            model_succeeded = True
            break

        except Exception as err:
            last_error = f"{type(err).__name__}: {err}"
            attempt_logs.append({
                'model_name': model_name,
                'attempt': attempt_idx,
                'user_cap': user_cap,
                'status': 'failed',
                'train_ratings': int(len(train_attempt_df)),
                'test_ratings': int(len(test_attempt_df)),
                'error': last_error
            })

    if not model_succeeded:
        model_results.append({
            'model_name': model_name,
            'status': 'failed',
            'attempts_used': len(config['user_caps']),
            'selected_user_cap': np.nan,
            'user_based': bool(sim_options['user_based']),
            'similarity': sim_options['name'],
            'train_users': np.nan,
            'train_items': np.nan,
            'train_ratings': np.nan,
            'test_ratings': np.nan,
            'rmse': np.nan,
            'mae': np.nan,
            'coverage': np.nan,
            'last_error': last_error if last_error is not None else 'Unknown error'
        })

metrics_df = pd.DataFrame(model_results).sort_values(['status', 'user_based', 'similarity', 'model_name']).reset_index(drop=True)
attempts_df = pd.DataFrame(attempt_logs).sort_values(['model_name', 'attempt']).reset_index(drop=True)

print(f"Modelos exitosos: {(metrics_df['status'] == 'success').sum()} / {len(metrics_df)}")
metrics_df

: 

## Exportar modelos
Guardar los 4 modelos entrenados (cosine/pearson, user-user/item-item) y su metadata.

In [3]:
from pathlib import Path
from surprise import dump
import json

model_dir = Path('artifacts')
model_dir.mkdir(parents=True, exist_ok=True)

export_rows = []

for model_name, model_obj in trained_models.items():
    row = metrics_df.loc[metrics_df['model_name'] == model_name].iloc[0]

    model_path = model_dir / f'model_{model_name}.pkl'
    metadata_path = model_dir / f'model_{model_name}_metadata.json'

    dump.dump(str(model_path), algo=model_obj)

    metadata = {
        'model_name': model_name,
        'backend': 'surprise',
        'algorithm': 'KNNBasic',
        'library': 'surprise',
        'status': str(row['status']),
        'similarity': str(row['similarity']),
        'user_based': bool(row['user_based']),
        'k_neighbors': int(config['k_neighbors']),
        'min_k': int(config['min_k']),
        'attempts_used': int(row['attempts_used']),
        'selected_user_cap': int(row['selected_user_cap']),
        'fallback_tiers': [int(v) for v in config['user_caps']],
        'max_test_eval': int(config['max_test_eval']),
        'train_users': int(row['train_users']),
        'train_items': int(row['train_items']),
        'train_ratings': int(row['train_ratings']),
        'test_ratings': int(row['test_ratings']),
        'rmse': float(row['rmse']),
        'mae': float(row['mae']),
        'coverage': float(row['coverage'])
    }

    with open(metadata_path, 'w', encoding='utf-8') as f:
        json.dump(metadata, f, ensure_ascii=False, indent=2)

    export_rows.append({
        'model_name': model_name,
        'status': str(row['status']),
        'attempts_used': int(row['attempts_used']),
        'selected_user_cap': int(row['selected_user_cap']),
        'backend': 'surprise',
        'model_path': str(model_path),
        'metadata_path': str(metadata_path)
    })

failed_models_df = metrics_df.loc[metrics_df['status'] != 'success', ['model_name', 'last_error']].copy()

if failed_models_df.empty:
    print('Todos los modelos se entrenaron y exportaron correctamente.')
else:
    print('Modelos que no se pudieron entrenar despues de todos los intentos:')
    print(failed_models_df.to_string(index=False))

export_df = pd.DataFrame(export_rows).sort_values('model_name').reset_index(drop=True) if export_rows else pd.DataFrame(
    columns=['model_name', 'status', 'attempts_used', 'selected_user_cap', 'backend', 'model_path', 'metadata_path']
)
export_df

Todos los modelos se entrenaron y exportaron correctamente.


,model_name,status,attempts_used,selected_user_cap,backend,model_path,metadata_path
0,item_item_cosine,success,1,8000,surprise,artifacts/model_item_item_cosine.pkl,artifacts/model_item_item_cosine_metadata.json
1,item_item_pearson,success,1,8000,surprise,artifacts/model_item_item_pearson.pkl,artifacts/model_item_item_pearson_metadata.json
2,user_user_cosine,success,1,8000,surprise,artifacts/model_user_user_cosine.pkl,artifacts/model_user_user_cosine_metadata.json
3,user_user_pearson,success,1,8000,surprise,artifacts/model_user_user_pearson.pkl,artifacts/model_user_user_pearson_metadata.json
